In [ ]:
from google.colab import files

uploaded = files.upload()

Saving synthetic_flight_passenger_data.csv to synthetic_flight_passenger_data.csv


In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


# ============================================================
# 2. LOAD THE DATASET
# ============================================================

# Read the CSV file into a Pandas DataFrame
df = pd.read_csv("synthetic_flight_passenger_data.csv")

# Display the first 5 rows
df.head()

# Check the number of rows and columns
df.shape

# Check missing values in every column
df.isnull().sum()

# Generate descriptive statistics for all columns
df.describe(include="all").T

# Display all column names
df.columns

# Check the data type of every column
df.dtypes

# Check for completely duplicated rows
df.duplicated().sum()


# ============================================================
# 3. CHECK AND CLEAN MISSING VALUES
# ============================================================

# Check the values in Frequent_Flyer_Status,
# including the missing values
df["Frequent_Flyer_Status"].value_counts(dropna=False)

# Inspect some departure-time values
df["Departure_Time"].head(10)
df["Departure_Time"].tail(10)

# Check how many unique departure timestamps exist
df["Departure_Time"].nunique()

# Test whether all departure-time values can be converted
# into valid datetime values
pd.to_datetime(
    df["Departure_Time"],
    errors="coerce"
).isna().sum()

# Convert Departure_Time from text/object to datetime
df["Departure_Time"] = pd.to_datetime(
    df["Departure_Time"],
    errors="coerce"
)

# Confirm that the column is now datetime
df["Departure_Time"].dtype


# Investigate rows where Frequent_Flyer_Status is missing
# to see whether there is an obvious pattern
df[df["Frequent_Flyer_Status"].isna()][
    [
        "Travel_Purpose",
        "Seat_Class",
        "Flight_Satisfaction_Score"
    ]
].head(10)

# Replace missing frequent-flyer status with a meaningful category
df["Frequent_Flyer_Status"] = df[
    "Frequent_Flyer_Status"
].fillna("Not a member")

# Check the updated category counts
df["Frequent_Flyer_Status"].value_counts()

# Confirm that there are no missing values left
df.isnull().sum()


# ============================================================
# 4. BASIC CATEGORY ANALYSIS
# ============================================================

# Count passengers for each airline
df["Airline"].value_counts()

# Count flights by flight status
df["Flight_Status"].value_counts()


# ============================================================
# 5. CHART 1 — NUMBER OF PASSENGERS BY AIRLINE
# ============================================================

# Count the number of passengers for each airline
airline_counts = df["Airline"].value_counts().reset_index()

# Rename the columns to make them easier to understand
airline_counts.columns = ["Airline", "Passengers"]

# Create a bar chart
fig = px.bar(
    x=airline_counts["Airline"],
    y=airline_counts["Passengers"],
    title="Number of Passengers by Airline",
    labels={
        "Airline": "Airline",
        "Passengers": "Number of Passengers"
    },
    color=airline_counts["Passengers"],
    color_continuous_scale=px.colors.sequential.Plasma
)

fig.show()


# ============================================================
# 6. CHART 2 — FLIGHT ACTIVITY BY MONTH
# ============================================================

# Extract the month and year from Departure_Time
df["Month"] = df["Departure_Time"].dt.to_period("M").astype(str)

# Count the number of flights in each month
monthly_flights = (
    df["Month"]
    .value_counts()
    .sort_index()
    .reset_index()
)

# Rename the columns
monthly_flights.columns = ["Month", "Flights"]

# Create a line chart
fig = px.line(
    x=monthly_flights["Month"],
    y=monthly_flights["Flights"],
    title="Number of Flights by Month",
    markers=True,
    color_discrete_sequence=["purple"],
    labels={
        "Month": "Month",
        "Flights": "Number of Flights"
    }
)

fig.show()


# ============================================================
# 7. CHART 3 — DISTRIBUTION OF FLIGHT DELAYS
# ============================================================

# Create a histogram to see how flight delays are distributed
fig = px.histogram(
    df,
    x="Delay_Minutes",
    nbins=30,
    title="Distribution of Flight Delays",
    color_discrete_sequence=["crimson"],
    labels={
        "Delay_Minutes": "Delay (Minutes)",
        "count": "Number of Flights"
    }
)

fig.show()


# ============================================================
# 8. CHART 4 — TICKET PRICE BY SEAT CLASS
# ============================================================

# Create a box plot to compare ticket-price distributions
# across different seat classes
fig = px.box(
    df,
    x="Seat_Class",
    y="Price_USD",
    title="Ticket Price by Seat Class",
    color_discrete_sequence=["coral"],
    labels={
        "Seat_Class": "Seat Class",
        "Price_USD": "Ticket Price (USD)"
    }
)

fig.show()


# ============================================================
# 9. CHART 5 — FLIGHT DELAY VS DISTANCE
# ============================================================

# Create a scatter plot to investigate whether
# flight distance is related to delay duration
fig = px.scatter(
    x=df["Distance_Miles"],
    y=df["Delay_Minutes"],
    title="Flight Delay vs. Distance in Miles",
    labels={
        "Distance_Miles": "Distance (Miles)",
        "Delay_Minutes": "Flight Delay (Minutes)"
    },
    color_discrete_sequence=["seagreen"],
    trendline="ols"
)

fig.show()


# ============================================================
# 10. CHART 6 — CORRELATION HEATMAP
# ============================================================

# Select numerical variables for correlation analysis
numeric_cols = [
    "Delay_Minutes",
    "Distance_Miles",
    "Price_USD",
    "Flight_Duration_Minutes",
    "Flight_Satisfaction_Score",
    "Booking_Days_In_Advance"
]

# Calculate the correlation between every pair
# of selected numerical variables
corr_matrix = df[numeric_cols].corr()

# Create the heatmap
fig = go.Figure(
    data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.columns,

        # Display correlation values inside the cells
        text=corr_matrix.round(2).values,
        texttemplate="%{text}",

        # Blue = negative, red = positive
        colorscale="RdBu",
        zmin=-1,
        zmax=1
    )
)

fig.update_layout(
    title="Correlation Between Numerical Flight Variables",
    xaxis_title="Variables",
    yaxis_title="Variables"
)

fig.show()


# ============================================================
# 11. INTERACTIVE CHART 1 — FLIGHT STATUS BY AIRLINE
# ============================================================

# Check the different airlines
df["Airline"].unique()

# Count flights for each combination of airline and status
airline_status = (
    df.groupby(
        ["Airline", "Flight_Status"]
    )
    .size()
    .reset_index(name="Flights")
)

# Display the first few rows
airline_status.head()

# Store the airline names
airlines = df["Airline"].unique()

# Create an empty Plotly figure
fig = go.Figure()

# Create one bar-chart trace for each airline
for airline in airlines:

    # Keep only data belonging to the current airline
    data = airline_status[
        airline_status["Airline"] == airline
    ]

    # Add a bar chart for that airline
    fig.add_trace(
        go.Bar(
            x=data["Flight_Status"],
            y=data["Flights"],
            name=airline,

            # Initially display only the first airline
            visible=(airline == airlines[0])
        )
    )


# Create the dropdown buttons
buttons = []

for i, airline in enumerate(airlines):

    # Initially hide all airline traces
    visibility = [False] * len(airlines)

    # Make the selected airline visible
    visibility[i] = True

    # Create a dropdown option
    buttons.append(
        dict(
            label=airline,
            method="update",

            # Show only the selected airline
            # and update the chart title
            args=[
                {"visible": visibility},
                {"title": f"Flight Status — {airline}"}
            ]
        )
    )


# Add the dropdown to the chart
fig.update_layout(
    title="Flight Status by Airline",
    xaxis_title="Flight Status",
    yaxis_title="Number of Flights",

    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True
        )
    ]
)

fig.show()
fig.write_html("flight_status_by_airline.html")

# ============================================================
# 12. INTERACTIVE CHART 2 — TICKET PRICE BY SEAT CLASS
# ============================================================

# Store the different seat classes
seat_classes = df["Seat_Class"].unique()

# Display the seat classes
seat_classes

# Create an empty Plotly figure
fig = go.Figure()

# Create one box-plot trace for each seat class
for seat_class in seat_classes:

    # Filter the dataset for the current seat class
    data = df[
        df["Seat_Class"] == seat_class
    ]

    # Add a box plot for that seat class
    fig.add_trace(
        go.Box(
            y=data["Price_USD"],
            name=seat_class,

            # Initially display only the first seat class
            visible=(seat_class == seat_classes[0])
        )
    )


# Create the dropdown buttons
buttons = []

for i, seat_class in enumerate(seat_classes):

    # Initially hide all seat-class traces
    visibility = [False] * len(seat_classes)

    # Make the selected seat class visible
    visibility[i] = True

    # Create a dropdown option
    buttons.append(
        dict(
            label=seat_class,
            method="update",

            # Show only the selected seat class
            # and update the title
            args=[
                {"visible": visibility},
                {"title": f"Ticket Price — {seat_class}"}
            ]
        )
    )


# Add the dropdown to the chart
fig.update_layout(
    title="Ticket Price by Seat Class",
    yaxis_title="Ticket Price (USD)",

    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True
        )
    ]
)

fig.show()
fig.write_html("ticket_price_by_seat_class.html")


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'flight_status_by_airline.html', 'synthetic_flight_passenger_data.csv', 'ticket_price_by_seat_class.html', 'sample_data']
